### Ajouter prix au m2 par maison

In [1]:
import os
import pandas as pd

data_dir = os.path.join('..', 'csv', 'STEP02')

# Process each CSV file in STEP02
for file in os.listdir(data_dir):
    if file.endswith('.csv'):
        path = os.path.join(data_dir, file)
        df = pd.read_csv(path)
        
        # Nettoyage numérique
        df['Taille'] = pd.to_numeric(df['Taille'].astype(str).str.replace(' ', ''), errors='coerce')
        df['Prix'] = pd.to_numeric(df['Prix'].astype(str).str.replace(' ', ''), errors='coerce')
        
        # Assuming columns 'Prix' (price) and 'Taille' (area in m²)
        # Calculate 'Prix au m2'
        df['Prix au m2'] = df['Prix'] / df['Taille']
        
        # Arrondir à l'entier
        df['Prix au m2'] = df['Prix au m2'].round(0)
        
        # Save the updated DataFrame back to the same file
        df.to_csv(path, index=False)

### Supprimer maisons avec "ferme" "rénover" "renover"

In [2]:
import re
import pandas as pd
import os

# Créer le répertoire STEP03 s'il n'existe pas
os.makedirs('../csv/STEP03', exist_ok=True)

# Charger tous les CSV STEP02
files = [
    '../csv/STEP02/STEP02_maisons_dept22.csv',
    '../csv/STEP02/STEP02_maisons_dept29.csv',
    '../csv/STEP02/STEP02_maisons_dept35.csv',
    '../csv/STEP02/STEP02_maisons_dept56.csv'
]

# Charger les DataFrames
dfs = [pd.read_csv(f, encoding='utf-8') for f in files]

# Fonction de filtrage
def filtrer_maisons(df):
    # On retire les lignes dont le titre contient "ferme", "rénover" ou "renover" (insensible à la casse)
    masque = ~df['Nom'].str.contains(r'ferme|rénover|renover', flags=re.IGNORECASE, na=False)
    return df[masque]

# Appliquer le filtre à chaque DataFrame de dfs
dfs_filtrés = [filtrer_maisons(d) for d in dfs]

# Sauvegarder dans les fichiers STEP03 et afficher le nombre de maisons supprimées
for i, df_filtré in enumerate(dfs_filtrés):
    dept_num = files[i].split('dept')[1].split('.csv')[0]
    nombre_avant = len(dfs[i])
    nombre_apres = len(df_filtré)
    nombre_supprimees = nombre_avant - nombre_apres
    output_path = f'../csv/STEP03/STEP03_maisons_dept{dept_num}.csv'
    df_filtré.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Département {dept_num}: {nombre_supprimees} maisons supprimées ({nombre_avant} → {nombre_apres})")
    print(f"Fichier sauvegardé : {output_path}")



Département 22: 87 maisons supprimées (4225 → 4138)
Fichier sauvegardé : ../csv/STEP03/STEP03_maisons_dept22.csv
Département 29: 121 maisons supprimées (5335 → 5214)
Fichier sauvegardé : ../csv/STEP03/STEP03_maisons_dept29.csv
Département 35: 129 maisons supprimées (5867 → 5738)
Fichier sauvegardé : ../csv/STEP03/STEP03_maisons_dept35.csv
Département 56: 147 maisons supprimées (5898 → 5751)
Fichier sauvegardé : ../csv/STEP03/STEP03_maisons_dept56.csv


### Supprimer les maisons doublons

In [3]:
import re
import pandas as pd

# Charger tous les CSV STEP03
files = [
    '../csv/STEP03/STEP03_maisons_dept22.csv',
    '../csv/STEP03/STEP03_maisons_dept29.csv',
    '../csv/STEP03/STEP03_maisons_dept35.csv',
    '../csv/STEP03/STEP03_maisons_dept56.csv'
]

# Charger les DataFrames
dfs = [pd.read_csv(f) for f in files]

# Colonnes à vérifier pour les doublons (toutes sauf 'Nom' et 'Lien')
colonnes_a_verifier = ['Code INSEE', 'Page Lien', 'Prix', 'Lieu', 'Taille', 'Taille_terrain', 'Pieces', 'Prix au m2']

total_supprimees = 0

for idx, df in enumerate(dfs):
    nombre_avant = len(df)

    # Afficher les maisons dont seules les colonnes 'Nom' et 'Lien' diffèrent (potentiels doublons)
    doublons = df[df.duplicated(subset=colonnes_a_verifier, keep=False)]
    if not doublons.empty:
        print(f"\nDépartement {files[idx].split('dept')[1].split('.csv')[0]} : {len(doublons)} doublons potentiels trouvés")
        # Afficher les groupes de doublons avec leurs titres et liens
        grouped = doublons.groupby(colonnes_a_verifier)
        for _, group in grouped:
            if len(group) > 1:
                # Afficher seulement Nom, Prix et Lien avec lien cliquable
                df_affichage = group[['Nom', 'Prix', 'Lien']].copy()
                print(df_affichage.to_string(index=False))
                print("\nLiens:")
                for _, row in df_affichage.iterrows():
                    print(f"- {row['Nom']}: {row['Lien']}")
    else:
        print(f"\nDépartement {files[idx].split('dept')[1].split('.csv')[0]} : Aucun doublon potentiel trouvé")

    # Supprimer les doublons (garder la première occurrence)
    df_sans_doublons = df.drop_duplicates(subset=colonnes_a_verifier, keep='first')
    nombre_apres = len(df_sans_doublons)
    nombre_supprimees = nombre_avant - nombre_apres

    if nombre_supprimees > 0:
        print(f"  → {nombre_supprimees} doublons supprimés ({nombre_avant} → {nombre_apres})")

    # Sauvegarder le fichier nettoyé
    df_sans_doublons.to_csv(files[idx], index=False)
    total_supprimees += nombre_supprimees

print(f"\n=== TOTAL : {total_supprimees} doublons supprimés au total ===")


Département 22 : 117 doublons potentiels trouvés
                  Nom     Prix                                                                              Lien
Maison 120m² à callac 219900.0 https://www.etreproprio.com/immobilier-23429848-vente-maison-120m-a-callac-callac
Maison 120m² à callac 219900.0 https://www.etreproprio.com/immobilier-23165537-vente-maison-120m-a-callac-callac

Liens:
- Maison 120m² à callac: https://www.etreproprio.com/immobilier-23429848-vente-maison-120m-a-callac-callac
- Maison 120m² à callac: https://www.etreproprio.com/immobilier-23165537-vente-maison-120m-a-callac-callac
                 Nom     Prix                                                                            Lien
Maison 138m² à dinan 546000.0 https://www.etreproprio.com/immobilier-23823118-vente-maison-138m-a-dinan-dinan
Maison 138m² à dinan 546000.0 https://www.etreproprio.com/immobilier-23817313-vente-maison-138m-a-dinan-dinan

Liens:
- Maison 138m² à dinan: https://www.etreproprio.com

### Outliers ville précise (pour info)

In [7]:
import pandas as pd

# Configurer pandas pour afficher les colonnes longues en entier
pd.set_option('display.max_colwidth', None)

# Demander à l'utilisateur la ville à chercher
ville = input("Entrez le nom de la ville à chercher: ")

# Charger tous les CSV STEP03
files = [
    '../csv/STEP03/STEP03_maisons_dept22.csv',
    '../csv/STEP03/STEP03_maisons_dept29.csv',
    '../csv/STEP03/STEP03_maisons_dept35.csv',
    '../csv/STEP03/STEP03_maisons_dept56.csv'
]

dfs = [pd.read_csv(f, encoding='utf-8') for f in files]
df = pd.concat(dfs, ignore_index=True)

# Filtrer les maisons pour la ville donnée
maisons_ville = df[df['Lieu'].str.contains(ville, na=False)].copy()

print(f"Maisons à {ville} ({len(maisons_ville)} au total):")
if len(maisons_ville) > 0:
    # Limiter l'affichage pour le tableau principal
    pd.set_option('display.max_colwidth', 50)
    display(maisons_ville.sort_values('Prix au m2'))
    # Remettre à None pour les prints suivants
    pd.set_option('display.max_colwidth', None)

    # Nettoyage pour calculs
    maisons_ville['Prix au m2'] = pd.to_numeric(maisons_ville['Prix au m2'], errors='coerce')

    # Moyenne avec outliers
    mean_ville = maisons_ville['Prix au m2'].mean()
    print(f"\nMoyenne {ville}: {mean_ville:.2f} €/m²")

    # --- Détection outliers avec IQR ---
    Q1_ville = maisons_ville['Prix au m2'].quantile(0.25)
    Q3_ville = maisons_ville['Prix au m2'].quantile(0.75)
    IQR_ville = Q3_ville - Q1_ville

    print(f"IQR - Q1: {Q1_ville:.2f}, Q3: {Q3_ville:.2f}, IQR: {IQR_ville:.2f}")

    lower_ville = Q1_ville - 1.5 * IQR_ville
    upper_ville = Q3_ville + 1.5 * IQR_ville

    outliers_ville = maisons_ville[(maisons_ville['Prix au m2'] < lower_ville) | (maisons_ville['Prix au m2'] > upper_ville)]
    print(f"Outliers selon IQR ({len(outliers_ville)}):")
    print(outliers_ville[['Prix au m2', 'Prix', 'Taille', 'Lien']].sort_values('Prix au m2'))

    # Moyenne sans outliers
    df_sans_outliers = maisons_ville[~maisons_ville.index.isin(outliers_ville.index)]
    mean_sans_outliers = df_sans_outliers['Prix au m2'].mean()
    print(f"Moyenne sans outliers: {mean_sans_outliers:.2f} €/m²")

else:
    print("Aucune maison trouvée dans le dataset pour cette ville.")

Maisons à Carantec (74 au total):


,Nom,Lien,Code INSEE,Page Lien,Prix,Lieu,Taille,Taille_terrain,Pieces,Prix au m2
4529,Maison carantec 8 pièce(s) 190 m2,https://www.etreproprio.com/immobilier-2237735...,29023,https://www.etreproprio.com/annonces/th.lc2902...,462000.0,Carantec 29660,190.0,2 000,8.0,2432.0
4389,Maison carantec 7 pièce(s) 201 m2,https://www.etreproprio.com/immobilier-2383006...,29023,https://www.etreproprio.com/annonces/th.lc2902...,499500.0,Carantec 29660,201.0,625,7.0,2485.0
4457,Maison carantec 7 pièce(s) 119 m2,https://www.etreproprio.com/immobilier-2252031...,29023,https://www.etreproprio.com/annonces/th.lc2902...,296800.0,Carantec 29660,119.0,916,7.0,2494.0
4452,Maison 6 pièce(s) 112 m2,https://www.etreproprio.com/immobilier-2348170...,29023,https://www.etreproprio.com/annonces/th.lc2902...,304500.0,Carantec 29660,112.0,1 336,6.0,2719.0
4384,Maison carantec 6 pièces 112 m²,https://www.etreproprio.com/immobilier-2387964...,29023,https://www.etreproprio.com/annonces/th.lc2902...,304500.0,Carantec 29660,112.0,1 390,6.0,2719.0
4451,A vendre. maison. carantec.,https://www.etreproprio.com/immobilier-2268962...,29023,https://www.etreproprio.com/annonces/th.lc2902...,307000.0,Carantec 29660,112.0,1 390,6.0,2741.0
4387,Maison carantec 172 m²,https://www.etreproprio.com/immobilier-2387964...,29023,https://www.etreproprio.com/annonces/th.lc2902...,485000.0,Carantec 29660,172.0,1 300,9.0,2820.0
4455,Maison 172m² à carantec,https://www.etreproprio.com/immobilier-2301677...,29023,https://www.etreproprio.com/annonces/th.lc2902...,485000.0,Carantec 29660,172.0,1 062,6.0,2820.0
4530,A vendre. maison/appartement. carantec.,https://www.etreproprio.com/immobilier-2228610...,29023,https://www.etreproprio.com/annonces/th.lc2902...,227900.0,Carantec 29660,80.0,NaN,3.0,2849.0
4393,A vendre. maison. carantec.,https://www.etreproprio.com/immobilier-2227983...,29023,https://www.etreproprio.com/annonces/th.lc2902...,430000.0,Carantec 29660,144.0,4 650,7.0,2986.0



Moyenne Carantec: 4938.75 €/m²
IQR - Q1: 3844.00, Q3: 4513.00, IQR: 669.00
Outliers selon IQR (22):
      Prix au m2        Prix  Taille  \
4529      2432.0    462000.0   190.0   
4389      2485.0    499500.0   201.0   
4457      2494.0    296800.0   119.0   
4452      2719.0    304500.0   112.0   
4384      2719.0    304500.0   112.0   
4451      2741.0    307000.0   112.0   
4455      2820.0    485000.0   172.0   
4387      2820.0    485000.0   172.0   
4383      5824.0   1456000.0   250.0   
4531      6435.0   1995000.0   310.0   
4458      6437.0    869000.0   135.0   
4599      7396.0  10850000.0  1467.0   
4532      7962.0   1250000.0   157.0   
4533      8714.0   1612000.0   185.0   
4597      9158.0   5495000.0   600.0   
4594      9214.0   3695000.0   401.0   
4593      9389.0   3690000.0   393.0   
4595      9674.0   4750000.0   491.0   
4591      9787.0   2300000.0   235.0   
4589     12822.0  10950000.0   854.0   
4596     12895.0   4900000.0   380.0   
4598     15306.0   

### Outliers département

In [11]:
# Calculer les outliers par ville pour un département donné
import pandas as pd
import os

# Configurer pandas pour limiter l'affichage des colonnes longues
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', None)

# Demander le département
dept = input("Entrez le numéro du département (22, 29, 35 ou 56): ")

# Charger le CSV correspondant
file_path = f'../csv/STEP03/STEP03_maisons_dept{dept}.csv'
df = pd.read_csv(file_path, encoding='utf-8')

# Obtenir les villes uniques
villes = df['Lieu'].unique()

# Liste pour stocker les résultats
resultats = []
# Liste pour stocker les dataframes nettoyés
dfs_clean = []

for ville in villes:
    maisons_ville = df[df['Lieu'] == ville].copy()
    
    if len(maisons_ville) > 0:
        # Nettoyage pour calculs
        maisons_ville['Prix au m2'] = pd.to_numeric(maisons_ville['Prix au m2'], errors='coerce')
        maisons_ville = maisons_ville.dropna(subset=['Prix au m2'])
        
        nombre_maisons = len(maisons_ville)
        moyenne_avant = maisons_ville['Prix au m2'].mean()
        
        if len(maisons_ville) >= 4:  # Au moins 4 points pour calculer IQR
            # --- Détection outliers avec IQR ---
            Q1_ville = maisons_ville['Prix au m2'].quantile(0.25)
            Q3_ville = maisons_ville['Prix au m2'].quantile(0.75)
            IQR_ville = Q3_ville - Q1_ville

            lower_ville = Q1_ville - 1.5 * IQR_ville
            upper_ville = Q3_ville + 1.5 * IQR_ville

            outliers_ville = maisons_ville[(maisons_ville['Prix au m2'] < lower_ville) | (maisons_ville['Prix au m2'] > upper_ville)]
            nombre_outliers = len(outliers_ville)
            
            # Moyenne après outliers
            df_sans_outliers = maisons_ville[~maisons_ville.index.isin(outliers_ville.index)]
            moyenne_apres = df_sans_outliers['Prix au m2'].mean() if len(df_sans_outliers) > 0 else 0
        else:
            nombre_outliers = 0
            moyenne_apres = moyenne_avant  # Pas d'outliers retirés
            df_sans_outliers = maisons_ville  # Garder tout
    else:
        nombre_maisons = 0
        nombre_outliers = 0
        moyenne_avant = 0
        moyenne_apres = 0
        df_sans_outliers = maisons_ville  # Vide
    
    resultats.append({
        'Ville': ville, 
        'Nombre de maisons': nombre_maisons, 
        'Nombre d\'outliers': nombre_outliers,
        'Moyenne avant outliers (€/m²)': round(moyenne_avant, 2),
        'Moyenne après outliers (€/m²)': round(moyenne_apres, 2)
    })
    
    # Ajouter le dataframe nettoyé
    dfs_clean.append(df_sans_outliers)

# Créer un DataFrame avec les résultats
df_outliers = pd.DataFrame(resultats)

# Trier par nombre d'outliers décroissant
df_outliers = df_outliers.sort_values('Nombre d\'outliers', ascending=False)

# Afficher le tableau complet
display(df_outliers)

# Créer le CSV nettoyé
df_clean = pd.concat(dfs_clean, ignore_index=True)
file_path_clean = f'../csv/STEP04/STEP04_maisons_dept{dept}.csv'
os.makedirs(os.path.dirname(file_path_clean), exist_ok=True)
df_clean.to_csv(file_path_clean, index=False, encoding='utf-8')
print(f"Fichier nettoyé sauvegardé : {file_path_clean}")

,Ville,Nombre de maisons,Nombre d'outliers,Moyenne avant outliers (€/m²),Moyenne après outliers (€/m²)
238,Vannes 56000,263,18,4448.44,4137.12
218,Sarzeau 56370,200,9,4698.00,4451.26
6,Auray 56400,73,7,3757.07,3639.56
89,Languidic 56440,85,7,2426.16,2454.64
95,Larmor-Plage 56260,54,7,4733.76,4702.85
7,Baden 56870,86,7,5086.58,4897.92
69,Guidel 56520,88,7,3938.74,3779.21
146,Ploemeur 56270,87,7,4526.11,4119.45
94,Larmor-Baden 56870,30,5,6049.07,5708.28
9,Baud 56150,87,5,4022.77,2057.28


Fichier nettoyé sauvegardé : ../csv/STEP04/STEP04_maisons_dept56.csv


### Vérification

In [12]:
import pandas as pd
import os

# Définir les départements à traiter
departements = ['22', '29', '35', '56']

# Stocker les résultats
resultats = []

for dept in departements:
    file_step02 = f'../csv/STEP02/STEP02_maisons_dept{dept}.csv'
    file_step04 = f'../csv/STEP04/STEP04_maisons_dept{dept}.csv'
    
    # Charger les DataFrames
    df02 = pd.read_csv(file_step02, encoding='utf-8')
    df04 = pd.read_csv(file_step04, encoding='utf-8')
    
    n02 = len(df02)
    n04 = len(df04)
    diff = n02 - n04
    
    resultats.append({
        'Département': dept,
        'STEP02': n02,
        'STEP04': n04,
        'Différence': diff
    })

# Afficher le tableau récapitulatif
df_resultats = pd.DataFrame(resultats)
display(df_resultats)

,Département,STEP02,STEP04,Différence
0,22,4225,3892,333
1,29,5335,4695,640
2,35,5867,5391,476
3,56,5898,5338,560
